# Interval audit: what do the retained predictions already determine?

**One question.** For the frozen dispersed instance, how many original weight values are still
consistent with the retained 8-bit logit codes? Not "how many bits did our decoder need" -
how many candidates actually survive.

**Why.** The reported 10.9% saving was produced with `erasures=16` frozen against `sparsity=64`,
so the burden `e + 2u = 16 + 2(64-c)` is at best 112 against a 128-check baseline. The experiment
could not have exceeded a 12.5% saving for any N. It measured the l1 localizer's 25% recall, not
the information in the records.

**What this notebook does.** The l1 solver is replaced by exact interval arithmetic. Under the
one-change-per-row (dispersed) model, the set of numerical changes to column `j` of row `v`
consistent with all N retained codes is an *interval*, obtained by intersecting N scalar
constraints. Intersect it with the public rho bound and with the BF16 value grid, and the exact
finite candidate set falls out.

**Four measurements.**

| # | Measures | Decides |
|---|---|---|
| A | Row-screen completeness | whether a row-locator code is usable |
| B | Exact per-row candidate counts at N=256 | how many bits are genuinely unresolved |
| C | Slope of `log2(count)` against `log2(N)` | the L-vs-N law |
| D | Budget ladder in bits | which rung the method is on |

**Predicted law.** `width(I) ~ WIDTH / (N * |h_j|)`, so candidate counts should fall with slope
about -1 in log-log and terminate at 1 once the interval is narrower than one BF16 step. A
plateau instead would mean a genuine floor. Both outcomes are informative; this measures which.

**Scope discipline.** This does not train anything, add a model, sweep output precision, or
implement a new decoder. It reads existing caches and counts. Roughly one GPU-hour including
feature/code cache reuse.

**Information hygiene.** Cells marked `EVALUATOR ONLY` construct the tamper and score the
results. The measurement path reads only the damaged head, the retained hidden states, the
retained codes, and public settings.

In [ ]:
# Run once if needed.
# %pip install -q "torch>=2.4" "transformers>=4.48" "datasets>=3.2" pandas matplotlib tqdm

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import gc, json, math, time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class Config:
    # Mirrors the frozen full-demo contract so the audited instance is identical.
    model_id: str = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    hf_cache_dir: str = "/scratch/bbjr/skarmakar/huggingface"
    model_revision: str | None = None
    dataset_revision: str | None = None
    cache_dir: str = "repair_runs/certificate_pilot"      # reuse existing feature/code caches
    out_dir: str = "repair_runs/interval_audit"
    pilot_seed: int = 17
    retained_sample_seed: int = 100_017                   # first locked test trial
    tamper_trial_id: int = 1000                           # predetermined fresh test trial
    N: int = 256
    sparsity: int = 64
    output_bits: int = 8
    fixed_logit_range: float = 64.0
    rho_multiple: float = 10.0
    tamper_fraction_of_bound: float = 0.9
    constraint_slack: float = 1e-4                        # declared FP32 numerical tolerance
    context_tokens: int = 256
    feature_batch: int = 24
    head_batch: int = 32
    cal_contexts: int = 96
    test_pool: int = 384
    behavior_contexts: int = 128
    # Audit-specific knobs.
    n_grid: tuple = (1, 2, 4, 8, 16, 32, 64, 128, 256)
    verify_cap: int = 200_000        # exact re-check only when a row has at most this many
    verify_chunk: int = 4096
    head_profile_rows: int = 512     # sampled untouched rows for the whole-head profile
    run_head_profile: bool = True
    run_concentrated_check: bool = True

CFG = Config()
assert torch.cuda.is_available(), "A CUDA GPU is required."
assert CFG.N == 256 and CFG.sparsity == 64 and CFG.output_bits == 8
DEVICE = torch.device("cuda")
torch.manual_seed(CFG.pilot_seed); np.random.seed(CFG.pilot_seed)
OUT = Path(CFG.out_dir); OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path(CFG.cache_dir); CACHE.mkdir(parents=True, exist_ok=True)
REPORT = {}
print({"cache_dir": str(CACHE), "out_dir": str(OUT)})

## Inputs

Rebuilds exactly the pilot's text pools, hidden-state cache and 8-bit code cache. If those
caches already exist under `cache_dir`, nothing is recomputed.

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", revision=CFG.dataset_revision)

def choose_texts(split, n, seed):
    texts = [x["text"].strip() for x in ds[split] if len(x["text"].strip()) >= 80]
    if len(texts) < n:
        raise RuntimeError(f"Only {len(texts)} usable {split} rows; need {n}")
    rng = np.random.default_rng(seed)
    return [texts[i] for i in rng.choice(len(texts), n, replace=False)]

texts = {
    "cal":  choose_texts("train",      CFG.cal_contexts,                          CFG.pilot_seed),
    "test": choose_texts("test",       CFG.test_pool + CFG.behavior_contexts,     CFG.pilot_seed + 2),
}
texts["test"] = texts["test"][:-CFG.behavior_contexts]
print({k: len(v) for k, v in texts.items()})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CFG.model_id, cache_dir=CFG.hf_cache_dir, revision=CFG.model_revision)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    CFG.model_id, cache_dir=CFG.hf_cache_dir, revision=CFG.model_revision,
    torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, attn_implementation="sdpa").eval().to(DEVICE)
assert not bool(getattr(model.config, "tie_word_embeddings", True)), "Expected an untied head."
V, D = model.lm_head.weight.shape
W = model.lm_head.weight.detach().clone()   # BF16 on GPU; tampered in place below

def hidden_states(items, name):
    path = CACHE / f"hidden_{name}.pt"
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=True)
    out = []
    for start in tqdm(range(0, len(items), CFG.feature_batch), desc=f"features:{name}"):
        enc = tokenizer(items[start:start + CFG.feature_batch], return_tensors="pt",
                        padding=True, truncation=True, max_length=CFG.context_tokens).to(DEVICE)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            h = model.model(**enc, use_cache=False, return_dict=True).last_hidden_state
        out.append(h[:, -1].float().cpu())   # left padding: last position is the final real token
    h = torch.cat(out); torch.save(h, path); return h

H = {k: hidden_states(v, k) for k, v in texts.items()}
assert all(torch.equal(h, h.to(torch.bfloat16).float()) for h in H.values()), \
    "Hidden-state cache is not exactly BF16; delete it and recompute."
weight_rms = float(W.float().square().mean().sqrt())
rho = CFG.rho_multiple * weight_rms
tamper_step = CFG.tamper_fraction_of_bound * rho
row_scale = W.float().abs().amax(dim=1).clamp_min(1e-12) / 127.0   # INT8 reference scales
del model; gc.collect(); torch.cuda.empty_cache()
print({"vocab": V, "hidden": D, "head_entries": V * D, "rho": rho, "tamper_step": tamper_step})

In [ ]:
R = float(CFG.fixed_logit_range)
WIDTH = 2 * R / (2 ** CFG.output_bits)

def qcode(z):
    """Public 8-bit output quantizer. Saturation is an error, not a silent clip."""
    if bool(((z < -R) | (z >= R)).any()):
        raise RuntimeError("Public logit range saturated; the record convention is violated.")
    return torch.floor((z + R) / WIDTH).to(torch.uint8)

def original_codes(h, name, head):
    path = CACHE / f"codes_{name}_w16_native_acc32_y{CFG.output_bits}_v4.pt"
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=True)
    chunks = []
    for start in tqdm(range(0, len(h), CFG.head_batch), desc=f"stored outputs:{name}"):
        hb = h[start:start + CFG.head_batch].to(DEVICE, dtype=torch.float32)
        chunks.append(qcode(hb @ head.T).cpu())
    x = torch.cat(chunks); torch.save(x, path); return x

Ctest = original_codes(H["test"], "test", W.float())
print({"retained_codes": tuple(Ctest.shape), "cell_width": WIDTH})

In [ ]:
# Public calibration rule fixing the tamper pools (same rule as the pilot and the demo).
with torch.inference_mode():
    hcal = H["cal"].to(DEVICE, dtype=torch.float32)
    logits = hcal @ W.float().T
    row_pool = torch.topk(torch.softmax(logits, dim=-1).mean(0), min(256, V)).indices.cpu().numpy()
    col_pool = torch.topk(H["cal"].square().mean(0), min(512, D)).indices.cpu().numpy()
    del logits, hcal; torch.cuda.empty_cache()

# NOTE the pool sizes. If the decoder may use this promise, the honest data-free location cost is
# log2 C(len(row_pool)*len(col_pool), s), not log2 C(V*D, s). Both are printed in Measurement D.
pool_entries = len(row_pool) * len(col_pool)
print({"row_pool": len(row_pool), "col_pool": len(col_pool),
       "pool_entries": pool_entries, "head_entries": V * D})

## EVALUATOR ONLY - rebuild and apply the frozen tamper

Reproduces the demo's predetermined test instance. `true_rows` / `true_cols` / `true_old` are used
only to score the measurements; nothing below feeds them into the interval computation.

In [ ]:
def make_frozen_tamper(layout="dispersed"):
    """EVALUATOR ONLY. Reproduces the demo's INT8-derived, rho-bounded BF16 change."""
    rng = np.random.default_rng(CFG.pilot_seed + 10_000 * CFG.tamper_trial_id
                                + (0 if layout == "dispersed" else 1))
    if layout == "dispersed":
        rows = rng.choice(row_pool, CFG.sparsity, replace=False).astype(np.int64)
        cols = rng.choice(col_pool, CFG.sparsity, replace=True).astype(np.int64)
    else:
        rows = np.repeat(rng.choice(row_pool), CFG.sparsity).astype(np.int64)
        cols = rng.choice(col_pool, CFG.sparsity, replace=False).astype(np.int64)
    signs = rng.choice([-1, 1], CFG.sparsity).astype(np.int64)
    rr = torch.as_tensor(rows, device=DEVICE); cc = torch.as_tensor(cols, device=DEVICE)
    scale = row_scale[rr]
    old_signed = torch.round(W[rr, cc].float() / scale).clamp(-127, 127).to(torch.int64)
    step = torch.round(torch.full_like(scale, tamper_step) / scale).clamp_min(1).to(torch.int64)
    step = torch.minimum(step, torch.floor(torch.full_like(scale, rho) / scale).to(torch.int64))
    sign = torch.as_tensor(signs, device=DEVICE)
    proposed = (old_signed + sign * step).clamp(-127, 127)
    proposed = torch.where(proposed == old_signed,
                           (old_signed - sign * step).clamp(-127, 127), proposed)
    if bool((proposed == old_signed).any()):
        raise RuntimeError("INT8 reference change vanished.")
    old = W[rr, cc].clone()
    new = (old.float() + (proposed - old_signed).float() * scale).to(torch.bfloat16)
    if bool((new == old).any()) or float((new.float() - old.float()).abs().max()) > rho:
        raise RuntimeError("BF16 tamper violates the frozen contract.")
    assert len(np.unique(rows * D + cols)) == CFG.sparsity
    return {"rows": rows, "cols": cols, "old": old, "new": new}

_tamper = make_frozen_tamper("dispersed")
_rr = torch.as_tensor(_tamper["rows"], device=DEVICE)
_cc = torch.as_tensor(_tamper["cols"], device=DEVICE)
with torch.no_grad():
    W[_rr, _cc] = _tamper["new"]                   # W is now the DAMAGED head
true_rows = _tamper["rows"]; true_cols = _tamper["cols"]
true_old = _tamper["old"].float().cpu().numpy()    # original values, for scoring only

rng = np.random.default_rng(CFG.retained_sample_seed)
order = rng.choice(len(H["test"]), CFG.N, replace=True)
retained_H = H["test"][order].to(DEVICE, dtype=torch.float32)          # (N, D)
retained_codes = Ctest[torch.as_tensor(order)].to(DEVICE)              # (N, V) uint8
print({"changed_weights": CFG.sparsity, "distinct_true_rows": len(np.unique(true_rows)),
       "retained_H": tuple(retained_H.shape), "retained_codes": tuple(retained_codes.shape)})

## Measurement A - row-screen completeness

The only thing the screen needs to deliver for a row-locator code to work: every changed row must
move at least one retained output code. `bad_rows` is already what the pilot's `location_scores`
returns; this reports its recall and its false-positive count.

In [ ]:
def current_codes(h):
    out = []
    for start in range(0, len(h), CFG.head_batch):
        out.append(qcode(h[start:start + CFG.head_batch] @ W.float().T))
    return torch.cat(out)

cur_codes = current_codes(retained_H)
mismatch = (retained_codes != cur_codes).any(0)
bad_rows = torch.nonzero(mismatch, as_tuple=False).flatten().cpu().numpy()

true_row_set = set(map(int, np.unique(true_rows)))
caught = sorted(true_row_set & set(map(int, bad_rows)))
false_rows = sorted(set(map(int, bad_rows)) - true_row_set)
REPORT["A_row_screen"] = {
    "n_bad_rows": int(len(bad_rows)),
    "n_true_rows": len(true_row_set),
    "n_caught": len(caught),
    "row_recall": len(caught) / max(1, len(true_row_set)),
    "n_false_rows": len(false_rows),
    "changed_output_cells": int((retained_codes != cur_codes).sum()),
    "screen_complete": len(caught) == len(true_row_set),
}
print(json.dumps(REPORT["A_row_screen"], indent=2))
if not REPORT["A_row_screen"]["screen_complete"]:
    print("\\nNOTE: the screen missed a changed row. A row-locator code needs error rather than "
          "erasure decoding on the row summaries, doubling its row cost.")

## Measurement B - exact candidate counts

For row $v$, let $c_i = z_v \cdot h_i$ be the **damaged** logit on sample $i$ and
$[l_i, u_i)$ the original cell named by the retained code. Hypothesising a single changed
column $j$ with $w_{vj} = z_{vj} + \delta$:

$$l_i \le c_i + \delta\, h_{ji} < u_i \quad\text{for every } i,$$

so with $a_i = l_i - c_i$ and $b_i = u_i - c_i$ each sample gives $\delta \in [a_i/h_{ji},
b_i/h_{ji})$ when $h_{ji} > 0$, the reversed interval when $h_{ji} < 0$, and a
feasibility test when $h_{ji} = 0$. Intersecting over $i$, clipping to $|\delta| \le \rho$, and
counting BF16 values in the result gives the exact candidate set.

Closed endpoints and the declared numerical slack make this an **outer** bound. Rows whose total
stays under `verify_cap` are then re-checked exactly by recomputing codes, so the reported
verified counts are exact.

In [ ]:
# Sorted table of every finite BF16 value, so counting is a searchsorted rather than bit twiddling.
_pat = torch.arange(65536, dtype=torch.int32)
_val = _pat.to(torch.int16).view(torch.bfloat16).float().numpy()
_finite = np.isfinite(_val)
_ordidx = np.argsort(_val[_finite], kind="stable")
BF16_VALUES = _val[_finite][_ordidx].astype(np.float32)          # ascending, includes -0.0 and 0.0
BF16_LABELS = _pat.numpy()[_finite][_ordidx].astype(np.int64)    # matching bit patterns
assert np.all(np.diff(BF16_VALUES) >= 0) and len(BF16_VALUES) == int(_finite.sum())
print({"finite_bf16_values": len(BF16_VALUES)})


def row_intervals(z, h, codes_row, slack):
    """Feasible delta interval per column for one row, under the one-change-per-row model.

    z: (D,) damaged row.  h: (N, D) retained features.  codes_row: (N,) retained ORIGINAL codes.
    Reads no original weights and no tamper. Returns (lo, hi, feasible) as numpy arrays.

    The bound is computed in float64 while the codes were produced in float32, so `slack` must
    cover that discrepancy for the interval to be a genuine OUTER bound. The self-check below
    verifies that on this machine's arithmetic; `verify_row` then makes the count exact.
    """
    cur = h @ z                                          # (N,) damaged logits
    q = codes_row.to(torch.float32)
    a = (-R + q * WIDTH) - cur - slack                   # original cell, relaxed by the slack
    b = (-R + (q + 1) * WIDTH) - cur + slack
    pos, neg = h > 0, h < 0                              # h is (N, D)
    zer = ~(pos | neg)
    safe = torch.where(zer, torch.ones_like(h), h)       # avoid 0/0 in the discarded branch
    inf = torch.tensor(float("inf"), device=h.device, dtype=h.dtype)
    lo_c = torch.where(pos, a[:, None] / safe, torch.where(neg, b[:, None] / safe, -inf))
    hi_c = torch.where(pos, b[:, None] / safe, torch.where(neg, a[:, None] / safe,  inf))
    lo, hi = lo_c.amax(0), hi_c.amin(0)
    # A zero feature cannot explain a sample whose damaged logit is outside the original cell.
    dead = (zer & ((a[:, None] > 0) | (b[:, None] < 0))).any(0)
    lo = torch.clamp(lo, min=-rho); hi = torch.clamp(hi, max=rho)
    feasible = (lo <= hi) & ~dead
    return (lo.double().cpu().numpy(), hi.double().cpu().numpy(), feasible.cpu().numpy())


def count_candidates(z, lo, hi, feasible):
    """BF16 values in [z+lo, z+hi] per feasible column. Upper bound: closed ends plus the slack."""
    zc = z.double().cpu().numpy()
    left  = np.searchsorted(BF16_VALUES, (zc + lo).astype(np.float32), side="left")
    right = np.searchsorted(BF16_VALUES, (zc + hi).astype(np.float32), side="right")
    counts = np.where(feasible, np.maximum(right - left, 0), 0).astype(np.int64)
    return counts, left, right


def verify_row(z, h, codes_row, left, right, counts, cap, chunk):
    """Exact count: keep only (column, label) pairs reproducing EVERY retained code.

    Sound because the interval is an outer bound, so nothing outside it can qualify.
    Returns (verified_count, per_column_counts) or (None, None) when the cap binds.
    """
    if int(counts.sum()) > cap:
        return None, None
    cur = h @ z
    per_col = np.zeros_like(counts)
    for j in np.nonzero(counts > 0)[0]:
        values = torch.as_tensor(BF16_VALUES[left[j]:right[j]], device=h.device, dtype=h.dtype)
        delta = values - z[j]
        hj = h[:, j]
        kept = 0
        for s in range(0, len(delta), chunk):
            zlog = cur[None, :] + delta[s:s + chunk, None] * hj[None, :]     # (K, N)
            inrange = ((zlog >= -R) & (zlog < R)).all(1)                      # saturation is illegal
            ok = torch.zeros(zlog.shape[0], dtype=torch.bool, device=h.device)
            if bool(inrange.any()):
                sub = torch.floor((zlog[inrange] + R) / WIDTH).to(torch.uint8)
                ok[inrange] = (sub == codes_row[None, :]).all(1)
            kept += int(ok.sum())
        per_col[j] = kept
    return int(per_col.sum()), per_col

### Self-check: is the declared slack actually an outer bound?

The interval is computed in float64; the retained codes were produced in float32. If the slack is
too small the interval can *exclude* candidates that genuinely reproduce every code, and the
"exact" counts below would silently undercount. This builds synthetic rows, brute-forces the true
candidate set, and asserts the interval contains it. It also reports how loose the slack is, which
is why verification rather than the raw interval carries the numbers below.

In [ ]:
def slack_self_check(slacks=(0.0, 1e-5, 1e-4, 1e-3), trials=40, Dt=16, Nt=32, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    rows = []
    for slack in slacks:
        g2 = torch.Generator(device="cpu").manual_seed(seed)
        under = tight = used = 0
        for _ in range(trials):
            z0 = (torch.randn(Dt, generator=g2) * 0.05).to(torch.bfloat16).float().to(DEVICE)
            h  = (torch.randn(Nt, Dt, generator=g2) * 0.6).to(torch.bfloat16).float().to(DEVICE)
            codes = qcode(h @ z0)
            j0 = int(torch.randint(Dt, (1,), generator=g2))
            d  = float(torch.empty(1).uniform_(0.02, 0.15, generator=g2)) * (1 if int(torch.randint(2,(1,),generator=g2)) else -1)
            zd = z0.clone(); zd[j0] = torch.tensor(z0[j0].item() + d).to(torch.bfloat16).float()
            if float(zd[j0]) == float(z0[j0]) or abs(float(zd[j0] - z0[j0])) > rho:
                continue
            used += 1
            lo, hi, feas = row_intervals(zd, h, codes, slack)
            counts, left, right = count_candidates(zd, lo, hi, feas)
            # Brute force over every BF16 value within rho of the damaged entry, per column.
            brute = 0
            for j in range(Dt):
                m = np.abs(BF16_VALUES - float(zd[j])) <= rho
                cand = torch.as_tensor(BF16_VALUES[m], device=DEVICE)
                if not len(cand):
                    continue
                zlog = (h @ zd)[None, :] + (cand - zd[j])[:, None] * h[None, :, j]
                inr = ((zlog >= -R) & (zlog < R)).all(1)
                if bool(inr.any()):
                    sub = torch.floor((zlog[inr] + R) / WIDTH).to(torch.uint8)
                    brute += int((sub == codes[None, :]).all(1).sum())
            outer = int(counts.sum())
            under += int(outer < brute)
            tight += int(outer == brute)
            assert counts[j0] > 0, "the true column was declared infeasible"
        rows.append({"slack": slack, "trials": used, "undercounts": under,
                     "exactly_tight": f"{tight}/{used}"})
    return pd.DataFrame(rows)

SC = slack_self_check()
print(SC.to_string(index=False))
chosen = SC[SC["slack"] == CFG.constraint_slack]
assert len(chosen) and int(chosen["undercounts"].iloc[0]) == 0, \
    (f"constraint_slack={CFG.constraint_slack} is not an outer bound on this arithmetic; "
     "raise it before trusting any count below.")
REPORT["S_slack_self_check"] = SC.to_dict(orient="records")
print(f"\nslack={CFG.constraint_slack} is a valid outer bound but rarely tight, which is why "
      "verify_row (exact re-check) carries the numbers below, not the raw interval counts.")

In [ ]:
rows_to_audit = np.array(sorted(true_row_set), dtype=np.int64)   # changed rows; scored below
records = []
for row in tqdm(rows_to_audit, desc="interval audit"):
    z = W[int(row)].float()
    codes_row = retained_codes[:, int(row)]
    lo, hi, feas = row_intervals(z, retained_H, codes_row, CFG.constraint_slack)
    counts, left, right = count_candidates(z, lo, hi, feas)
    verified, per_col = verify_row(z, retained_H, codes_row, left, right, counts,
                                   CFG.verify_cap, CFG.verify_chunk)
    exact = verified is not None
    eff_counts = per_col if exact else counts
    # EVALUATOR ONLY scoring.
    k = int(np.flatnonzero(true_rows == row)[0])
    tc, tv = int(true_cols[k]), float(true_old[k])
    zc = z.double().cpu().numpy()
    in_interval = bool(counts[tc] > 0 and (zc[tc] + lo[tc]) <= tv <= (zc[tc] + hi[tc]))
    records.append({
        "row": int(row),
        "exact": exact,
        "candidates": int(eff_counts.sum()),                 # verified when exact, else upper bound
        "candidates_interval_upper": int(counts.sum()),
        "feasible_columns": int((eff_counts > 0).sum()),
        "column_pinned": bool((eff_counts > 0).sum() == 1),
        "fully_pinned": bool(eff_counts.sum() == 1),
        "true_column": tc,
        "true_column_survives": bool(eff_counts[tc] > 0),
        "true_value_in_interval": in_interval,
    })

B = pd.DataFrame(records)
B.to_csv(OUT / "per_row_candidates.csv", index=False)
n_exact = int(B["exact"].sum())
REPORT["B_candidates"] = {
    "rows_audited": len(B),
    "rows_counted_exactly": n_exact,
    "rows_reported_as_upper_bound": len(B) - n_exact,
    "true_column_survives_all_rows": bool(B["true_column_survives"].all()),
    "true_value_in_interval_all_rows": bool(B["true_value_in_interval"].all()),
    "rows_with_column_pinned": int(B["column_pinned"].sum()),
    "rows_fully_pinned": int(B["fully_pinned"].sum()),
    "median_feasible_columns": float(B["feasible_columns"].median()),
    "median_candidates": float(B["candidates"].median()),
    "max_candidates": int(B["candidates"].max()),
    "residual_bits_sum_log2": float(np.log2(np.maximum(B["candidates"].values, 1)).sum()),
    "residual_bits_is_upper_bound": bool(n_exact < len(B)),
}
print(json.dumps(REPORT["B_candidates"], indent=2))
print()
print(B.head(12).to_string(index=False))
assert B["true_column_survives"].all() and B["true_value_in_interval"].all(), \
    "The true original was excluded: the slack or the cell convention is wrong, not the theory."

**Read this as:** `residual_bits_sum_log2` is $\sum_v \log_2(\text{candidates}_v)$, the
information still missing about *values and columns* once the rows are known. Compare it to the
3,534 bits the current certificate spends. The row identities themselves are supplied free by
Measurement A.

## Measurement C - the L-versus-N law

Recompute the intervals from the first $N$ retained samples only. The predicted law is
$\text{width}(I) \sim \text{WIDTH}/(N\,|h_j|)$, so `log2(candidates)` should fall with slope about
$-1$ against `log2(N)` and terminate at 0 once the interval is narrower than one BF16 step. A
plateau above 0 instead would indicate a genuine floor.

In [ ]:
sweep = []
for n in tqdm(CFG.n_grid, desc="N sweep"):
    hN = retained_H[:n]
    for row in rows_to_audit:
        z = W[int(row)].float()
        codes_row = retained_codes[:n, int(row)]
        lo, hi, feas = row_intervals(z, hN, codes_row, CFG.constraint_slack)
        counts, left, right = count_candidates(z, lo, hi, feas)
        verified, _ = verify_row(z, hN, codes_row, left, right, counts,
                                 CFG.verify_cap, CFG.verify_chunk)
        sweep.append({"N": n, "row": int(row), "exact": verified is not None,
                      "candidates": int(verified if verified is not None else counts.sum())})

C = pd.DataFrame(sweep)
C.to_csv(OUT / "candidates_vs_N.csv", index=False)
piv = C.pivot(index="N", columns="row", values="candidates").clip(lower=1)
exact_frac = C.groupby("N")["exact"].mean()
median = piv.median(axis=1)

# Fit only where the median is exact (verification did not hit the cap) and not yet saturated at 1.
mask = (median > 1) & (exact_frac >= 0.5)
if int(mask.sum()) >= 3:
    slope, intercept = np.polyfit(np.log2(median.index.values[mask].astype(float)),
                                  np.log2(median.values[mask]), 1)
else:
    slope = intercept = float("nan")
REPORT["C_law"] = {
    "median_candidates_by_N": {int(k): float(v) for k, v in median.items()},
    "fraction_exact_by_N": {int(k): float(v) for k, v in exact_frac.items()},
    "fitted_slope_log2count_vs_log2N": float(slope),
    "fit_used_N": [int(x) for x in median.index.values[mask]],
    "predicted_slope": -1.0,
    "N_at_which_median_reaches_1": (int(median[median <= 1].index.min())
                                    if (median <= 1).any() else None),
    "caveat": "points where fraction_exact < 1 are interval upper bounds, not exact counts",
}
print(json.dumps(REPORT["C_law"], indent=2))

In [ ]:
# One panel. The 64 rows are a distribution, not 64 identities, so they are drawn as a faint
# ensemble with the median carrying the reading; the dashed line is the predicted -1 slope.
INK, MUTED, ACCENT, GRID = "#1c1c1c", "#8a8a8a", "#2f6f9f", "#e2e2e2"
fig, ax = plt.subplots(figsize=(6.4, 4.4), dpi=140)
Nv = piv.index.values.astype(float)
for col in piv.columns:
    ax.plot(Nv, piv[col].values, color=MUTED, alpha=0.16, linewidth=1.0, zorder=1)
ax.plot(Nv, median.values, color=ACCENT, linewidth=2.0, marker="o", markersize=5,
        zorder=3, label="median over changed rows")
if np.isfinite(slope):
    ref = median.iloc[0] * (Nv / Nv[0]) ** (-1.0)
    ax.plot(Nv, np.maximum(ref, 1), color=INK, linewidth=1.5, linestyle="--",
            zorder=2, label="predicted slope $-1$")
ax.set_xscale("log", base=2); ax.set_yscale("log", base=2)
ax.set_xlabel("retained predictions $N$")
ax.set_ylabel("surviving original values per changed row")
ax.set_title(f"Candidates left by the retained record  (fitted slope {slope:.2f})", loc="left")
ax.grid(True, which="major", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(MUTED)
ax.tick_params(colors=MUTED); ax.yaxis.label.set_color(INK); ax.xaxis.label.set_color(INK)
ax.legend(frameon=False, labelcolor=INK)
fig.tight_layout(); fig.savefig(OUT / "candidates_vs_N.png", bbox_inches="tight")
plt.show()

## Measurement D - the budget ladder

Each rung is a certificate scheme; the measurements above say which are actually reachable on
this instance. Rungs marked unmet are printed with the condition that failed.

In [ ]:
P31, P17 = 2 ** 31 - 1, 2 ** 17 - 1
assert all(P17 % f for f in range(2, int(P17 ** 0.5) + 1)), "2**17-1 must be prime."
assert P17 > max(V, D, 65535), "Row-locator field must exceed vocab, hidden size and the alphabet."

s = CFG.sparsity
n_bad = REPORT["A_row_screen"]["n_bad_rows"]
screen_ok = REPORT["A_row_screen"]["screen_complete"]
cols_pinned = int(B["column_pinned"].sum()) == len(B)
residual_bits = REPORT["B_candidates"]["residual_bits_sum_log2"]

ladder = [
    ("data-free full-head Reed-Solomon (2s checks, p=2^31-1)", 2 * s * 31, True, ""),
    ("as reported: l1 top-16 flags + RS (114 checks)", 114 * 31, True, ""),
    ("row-summary code + row screen (2|E| checks, p=2^17-1)", 2 * n_bad * 17, screen_ok,
     "" if screen_ok else "row screen missed a changed row"),
    ("interval localization + RS values (|E| checks, p=2^31-1)", n_bad * 31,
     screen_ok and cols_pinned,
     "" if (screen_ok and cols_pinned) else "some row has more than one feasible column"),
    ("row locators + interval column ID (|E| checks, p=2^17-1)", n_bad * 17,
     screen_ok and cols_pinned,
     "" if (screen_ok and cols_pinned) else "some row has more than one feasible column"),
    ("information still unresolved after the record (sum log2 candidates)",
     int(math.ceil(residual_bits)), True, "counting bound given the rows; not a construction"),
]
Dtab = pd.DataFrame(ladder, columns=["scheme", "bits", "condition_met", "note"])
Dtab["bytes"] = np.ceil(Dtab["bits"] / 8).astype(int)
Dtab["vs_data_free"] = (1 - Dtab["bits"] / (2 * s * 31)).map(lambda x: f"{100*x:.1f}%")
Dtab.to_csv(OUT / "budget_ladder.csv", index=False)
REPORT["D_ladder"] = Dtab.to_dict(orient="records")

# Honest location accounting under the two readings of the tamper pools.
loc_full = float(np.log2(math.comb(V * D, s)))
loc_pool = float(np.log2(math.comb(pool_entries, s)))
REPORT["D_location_bits"] = {"full_head": loc_full, "public_pool": loc_pool,
                             "note": "if the pool is a promise the decoder may use, the honest "
                                     "data-free baseline uses the smaller figure"}
print(Dtab.to_string(index=False))
print()
print(json.dumps(REPORT["D_location_bits"], indent=2))

## Optional - whole-head ambiguity profile

The same reduction on untouched rows, sampled rather than exhaustive. It answers "how much
ambiguity would a change here leave?" for typical coordinates rather than for the
calibration-selected tamper pools, and it is the packing family behind a lower bound: these
candidates share the retained transcript **and** a common damaged checkpoint.

Sampling 512 rows estimates the distribution at 1/250th of the full-head cost. Set
`run_head_profile=False` to skip.

In [ ]:
if CFG.run_head_profile:
    rng = np.random.default_rng(CFG.pilot_seed + 7)
    untouched = np.setdiff1d(np.arange(V), np.array(sorted(true_row_set)))
    sample_rows = rng.choice(untouched, min(CFG.head_profile_rows, len(untouched)), replace=False)
    prof = []
    for row in tqdm(sample_rows, desc="head profile"):
        z = W[int(row)].float()
        codes_row = retained_codes[:, int(row)]
        lo, hi, feas = row_intervals(z, retained_H, codes_row, CFG.constraint_slack)
        counts, left, right = count_candidates(z, lo, hi, feas)
        verified, _ = verify_row(z, retained_H, codes_row, left, right, counts,
                                 CFG.verify_cap, CFG.verify_chunk)
        prof.append({"row": int(row), "exact": verified is not None,
                     "feasible_columns": int((counts > 0).sum()),
                     "candidates": int(verified if verified is not None else counts.sum())})
    Pf = pd.DataFrame(prof); Pf.to_csv(OUT / "head_profile.csv", index=False)
    q = Pf["candidates"].clip(lower=1)
    REPORT["E_head_profile"] = {
        "rows_sampled": len(Pf),
        "fraction_exact": float(Pf["exact"].mean()),
        "median_candidates": float(q.median()),
        "p90_candidates": float(q.quantile(0.90)),
        "median_log2": float(np.log2(q.median())),
        "note": "untouched rows: these candidates are alternative originals sharing this "
                "transcript and a common damaged checkpoint, so s times the median log2 "
                "lower-bounds any encoder for an s-sparse tamper on comparable rows",
    }
    print(json.dumps(REPORT["E_head_profile"], indent=2))
    print(f"\nimplied lower bound for s={CFG.sparsity} comparable rows: "
          f"{CFG.sparsity * REPORT['E_head_profile']['median_log2']:.0f} bits")
else:
    print("skipped")

## Optional - why concentrated tampers behave differently

Concentrated puts all 64 changes in one row, so the one-change-per-row hypothesis is false there.
The intervals should come back **empty for every column**: the record rejects the single-column
explanation outright. That is the cheap, direct version of the claim that dispersed gives `N`
constraints per unknown while concentrated gives `N/s`.

In [ ]:
if CFG.run_concentrated_check:
    W_saved = W[_rr, _cc].clone()
    with torch.no_grad():
        W[_rr, _cc] = _tamper["old"]                       # restore the original entries
    conc = make_frozen_tamper("concentrated")
    crr = torch.as_tensor(conc["rows"], device=DEVICE)
    ccc = torch.as_tensor(conc["cols"], device=DEVICE)
    with torch.no_grad():
        W[crr, ccc] = conc["new"]
    crow = int(conc["rows"][0])
    z = W[crow].float()
    codes_row = retained_codes[:, crow]
    lo, hi, feas = row_intervals(z, retained_H, codes_row, CFG.constraint_slack)
    counts, _, _ = count_candidates(z, lo, hi, feas)
    REPORT["F_concentrated"] = {
        "row": crow,
        "changes_in_row": int(len(conc["cols"])),
        "feasible_columns_under_one_change_model": int((counts > 0).sum()),
        "single_column_hypothesis_rejected": bool((counts > 0).sum() == 0),
        "note": "dispersed gives N constraints per unknown; concentrated gives N/s, so the "
                "single-column explanation should be rejected outright here",
    }
    print(json.dumps(REPORT["F_concentrated"], indent=2))
    with torch.no_grad():                                  # restore the dispersed instance
        W[crr, ccc] = conc["old"]
        W[_rr, _cc] = W_saved
else:
    print("skipped")

In [ ]:
REPORT["config"] = {k: (list(v) if isinstance(v, tuple) else v)
                    for k, v in CFG.__dict__.items()}
REPORT["provenance"] = {
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
(OUT / "interval_audit_report.json").write_text(json.dumps(REPORT, indent=2))
print(json.dumps({k: REPORT[k] for k in REPORT if k.startswith(("A_", "B_", "C_", "E_", "F_"))},
                 indent=2)[:4000])

## Decision rule

Read `interval_audit_report.json` against these three, and pick one:

1. **`rows_fully_pinned` is most of 64** - the retained record already determines the repair.
   The paper is "the records pin the values; the residual budget is row identity, which the
   mismatch screen supplies free," with Measurement C's slope as the mechanism. Write it.
2. **Candidates shrink at slope about -1 but stay above 1** - the honest `L(N)` law, with
   Measurement D naming the construction that captures it. Write that.
3. **Candidates stay large and flat** - a real floor. Find out whether the cause is feature
   coverage, precision, or within-row combinations; the head profile then has a converse target,
   and exact symbol recovery is arguably the wrong ask.

Whichever holds, **stop measuring after this.** Do not add model scales, output-precision sweeps,
adversarial tampers, or a behavioral decoder before the ladder rung is settled. And do not run
more trials at the old frozen configuration: at `erasures=16` against `sparsity=64` it cannot
exceed a 12.5% saving for any `N`.